<a href="https://colab.research.google.com/github/GurneeshBudhiraja/ArangoDB-Hackathon/blob/main/ArangoDB_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ArangoDB Hackathon :: Synthea_P100 Dataset Agent

### Install the required dependencies

In [1]:
# Required packages
!pip install nx-arangodb
!pip install arango-datasets
!pip install google-genai
# !pip install cugraph-cu12 --extra-index-url=https://pypi.nvidia.com # Requires the Google Colab runtime set to GPU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which

### Import the packages/modules

In [2]:
# Import required modules
import json
import pprint
from google.colab import userdata
from pydantic import BaseModel
from enum import Enum
from typing import Optional, List
import pandas as pd


import networkx as nx
import nx_arangodb as nxadb
# import cudf
# import cugraph as cg  # GPU-accelerated graph processing


from arango import ArangoClient
from arango_datasets import Datasets


# Gemini SDK Packages
from google import genai

[08:50:36 +0000] [INFO]: NetworkX-cuGraph is available.
INFO:nx_arangodb:NetworkX-cuGraph is available.
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:502: UserWarning: <built-in function any> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


### Cuda Version and GPU Information

In [3]:
# Gets the cuda version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [4]:
# Gets the GPU details => would only work when the runtime is set to GPU in Google Colab
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Application Code

In [5]:
# Imports the secrets from the Colab notebook
ARANGO_HOST = userdata.get("ARANGO_HOST")
ARANGO_PASSWORD = userdata.get("ARANGO_PASSWORD")
ARANGO_USERNAME = userdata.get("ARANGO_USERNAME")
GEMINI_API = userdata.get("GEMINI_API_KEY")

# Gemini models
GEMINI_FLASH_MODEL = "gemini-2.0-flash"
GEMINI_FLASH_LITE_MODEL = "gemini-2.0-flash-lite"
GEMINI_PRO_MODEL = "gemini-1.5-pro"


# Creates a db connection
arango_client = ArangoClient(hosts=ARANGO_HOST).db(username=ARANGO_USERNAME, password=ARANGO_PASSWORD, verify=True)

# Creates the genai instance
gemini_client = genai.Client(
    api_key=GEMINI_API
)

In [6]:
# Creates the dataset connection using the db object
datasets = Datasets(arango_client)

DATASET_NAME = "SYNTHEA_P100"

# Conditionally Loads the Synthea P100 dataset in Arango
if not arango_client.has_graph(DATASET_NAME):
  datasets.load(dataset_name=DATASET_NAME)
else:
  print(f"{DATASET_NAME} is already in ArangoDB.")

Output()

SYNTHEA_P100 is already in ArangoDB.


In [7]:
# Connects with the Graph in ArangoDB
graph = None
if arango_client.has_graph(DATASET_NAME):
  graph = nxadb.Graph(name=DATASET_NAME, db=arango_client)
else:
  print("Graph does not exist in Arango DB")

print(graph)

[08:50:41 +0000] [INFO]: Graph 'SYNTHEA_P100' exists.
INFO:nx_arangodb:Graph 'SYNTHEA_P100' exists.
[08:50:41 +0000] [INFO]: Default node type set to 'allergies'
INFO:nx_arangodb:Default node type set to 'allergies'


Graph named 'SYNTHEA_P100' with 145514 nodes and 311701 edges


### System Prompts and LLM output schemas

In [82]:
prompts_dict = {
    #  System prompt for the starter function
    "starter_prompt":"""
        Your main job is to find the relevance of the prompt or suggest the proper agent for solving the user_query.
        For finding the relevance, you are given with the Synthea_P100 dataset in ArangoDB. If the initial user query is not related to inquiring the information about this dataset, they you would deny the request and will not entertain it further. For instance, the irrelevant user_queries that are beyond your scope is but not limited to, What is your name?, What is the weather outside?,  What is the capital of Canada? or anything that is not related to the Synthea_P100 dataset. The examples of relevant queries are: What is the name of the patient whose patient id is patients/xxxx-yyyy-zzzz, How many patients have allergies to Latex and groundnuts, and anything that is related to the Synthea_P100 dataset. The data is stored in the form of graph and collections in ArangoDB and here are the Vertices/Nodes and Edges in the Synthea_P100 dataset:
            Vertex Collections:
                allergies
                careplans
                conditions
                devices
                encounters
                imaging_studies
                immunizations
                medications
                observations
                organizations
                patients
                payers
                procedures
                providers
                supplies

            Edge Collections:
                encounters_to_allergies
                encounters_to_careplans
                encounters_to_conditions
                encounters_to_devices
                encounters_to_imaging_studies
                encounters_to_immunizations
                encounters_to_medications
                encounters_to_observations
                encounters_to_procedures
                encounters_to_supplies
                organizations_to_encounters
                organizations_to_providers
                patients_to_allergies
                patients_to_careplans
                patients_to_conditions
                patients_to_devices
                patients_to_encounters
                patients_to_imaging_studies
                patients_to_immunizations
                patients_to_medications
                patients_to_observations
                patients_to_procedures
                patients_to_supplies
                payers_to_encounters
                payers_to_medications
                providers_to_encounters

        If anything that is asked can not be found even in the Synthea_P100 dataset, in that cases too you will deny the request.

        Another thing that you have to do is, find the right agent for solving the user queries. So, there are 3 agents named as aql, networkX and hybrid. Analyze whether it is best to solve the user query using the AQL, NetworkX algorithm or hybrid(involves using both AQL and NetworkX queries) and select the agent accordingly. Please note that whatever you suggest should be the most optimized approach and the best approach among all three.
    """,
    #  System prompt for the check memory
    "check_memory_prompt" : "Go throught the memory list that will be provided and try to find the answer to the user question. It is alright that if the answer to the user question is not present. Make sure to properly check if the answer to the user question is present in the memory list that will be provided. Also, please make sure that if you do find the info in the memory, reply the user in the natural language that should not sound robotic and if you got a question that is not related to the Synthea_P100 dataset, deny that requests too positively. Please make sure if the same or another version of the given user query is mentioned in the memory, then only you would answer the question. If you do not have the same or another same version of the question in the memory in that cases you would not use the memory to answer the user question.",
    #  System prompt for the AQL agent
    "aql_agent_prompt":"""
        You are the smart agent whose main job is to use the required tools to help answer the user query when the data that is stored is Synthea_P100. I will also provide you the list that contains past asked user questions and answers made by you. The format of the history list would be a list containing the dictionary like below:
            {
                "user_query":str,
                "agent_response":str
            }
        If you think that the complete whole answer that the user wants is in the history, you can reply from there otherwise follow the below instructions.

        Please make sure to always run the chunker tool that would help break the complex user queries into simple natural language queries. The chunker tool would return list of queries that you have use the proper tool to convert the simpler queries into the AQL queries and then also execute those AQL queries using the assigned tool. You will only rely on the tools that are provided to you to maintain the correctness and working of the application running.

        At the end, for all the data that you collected from different tools during the opearation, combine that result into the natural language and present to the user. Make sure the final response should give an idea at the end what sort data has been fetched by you and what data is missing.

        Also, before running any tool check that whether the user query is valid and would require fetching the data from the Synthea_P100 dataset. If it is not, you can deny the request.
    """,

    # System prompt for the NetworkX agent
    "networkX_agent_prompt":"""
        You are the networkX agent whose main job is to find the answer to the user query using the grapgh algorithnms NetworkX algorithms on the Synthea_P100 dataset stored in the ArangoDB. You are the agent that has access to the different tools that would help you execute networkX queries. If you think that a certain query is not related to the Synthea_P100 dataset, you can deny the request.

        You will only entertain the following graph queries and use the proper tool to generate/implement graph queries that would be best based on the user_query:
            1. Degree Centrality:
            2. Betweenness Centrality:
            3. Label Propagation (for Community Detection):
            4. Louvain Modularity (for Community Detection):
            5. Shortest Path:
        Once the AQL query has been generated you will use the proper tool to get the results from the ArangoDB.

        Also these are the collections in the Synthea_P100 dataset(stored in ArangoDB).
        Vertex Collections:
            allergies
            careplans
            conditions
            devices
            encounters
            imaging_studies
            immunizations
            medications
            observations
            organizations
            patients
            payers
            procedures
            providers
            supplies

        Edge Collections:
            encounters_to_allergies
            encounters_to_careplans
            encounters_to_conditions
            encounters_to_devices
            encounters_to_imaging_studies
            encounters_to_immunizations
            encounters_to_medications
            encounters_to_observations
            encounters_to_procedures
            encounters_to_supplies
            organizations_to_encounters
            organizations_to_providers
            patients_to_allergies
            patients_to_careplans
            patients_to_conditions
            patients_to_devices
            patients_to_encounters
            patients_to_imaging_studies
            patients_to_immunizations
            patients_to_medications
            patients_to_observations
            patients_to_procedures
            patients_to_supplies
            payers_to_encounters
            payers_to_medications
            providers_to_encounters

        Only use the required tools to achieve the end result and the end reply to the user query in a natural language providing a clear answer to the user query.
        Do not mention about the process you took to reach the final answer. The user should not know what sort of tools you used and anything related to the code.
    """

}


In [27]:
class AgentChoice(str, Enum):
    AQL = "aql"
    NETWORKX = "networkX"
    HYBRID = "hybrid"
    NULL = "null"

class StarterFunctionSchema(BaseModel):
    is_relevant: bool
    not_relevant_message: Optional[str]
    is_agent_recommended: bool
    agent_choice: AgentChoice


class CheckMemorySchema(BaseModel):
    is_present:bool
    memory_answer: Optional[str]

# AQL Agent and  AQL Agent's Tool Schema
class AQLSchema(BaseModel):
  query:str

class ChunkerSchema(BaseModel):
  queries: str

### Checks the user query's relevance and suggests the right agent accordingly

#### Initializes the memory array to store the chat messages and creates the basic llm



In [28]:
agent_memory = []

#### Function to check user query's relevance, uses memory to answer the repeated questions, and suggests the right agent

In [29]:
def starter_function(user_query:str):
    json_starter_reponse = gemini_client.models.generate_content(
        model=GEMINI_FLASH_LITE_MODEL,
        config = {
            "system_instruction": prompts_dict["starter_prompt"],
            "response_mime_type": "application/json",
            "response_schema":StarterFunctionSchema
        },
        contents=[f"user_query: {user_query}. In the final answer do not mention about the sources, just answer the user in the form of natural language."]
    )
    starter_response = json.loads(json_starter_reponse.text)
    print(f"================= Currently at : : Starter Function =====================\n")
    return starter_response


def check_memory(user_query:str):
    json_check_memory_response = gemini_client.models.generate_content(
        model=GEMINI_FLASH_MODEL,
        config = {
            "system_instruction": prompts_dict["check_memory_prompt"],
            "response_mime_type": "application/json",
            "response_schema":CheckMemorySchema
        },
        contents=f"user_query: {user_query} and the memory list is {agent_memory}"
    )
    print(f"================= Currently at : : Check Memory Function =====================\n")
    check_memory_response = json.loads(json_check_memory_response.text)
    return check_memory_response



### AQL Agent






#### AQL Agent Specific Prompts

In [30]:
aql_tool_system_prompts = {
    # Prompt for the aql_generator tool
    "aql_generator": """
        You are the ai assistant who will generate AQL queries. The ArangoDB has Synthea_P100 dataset with the following vertex and edge collections:

        Vertex Collections:
            allergies
            careplans
            conditions
            devices
            encounters
            imaging_studies
            immunizations
            medications
            observations
            organizations
            patients
            payers
            procedures
            providers
            supplies

        Edge Collections:
            encounters_to_allergies
            encounters_to_careplans
            encounters_to_conditions
            encounters_to_devices
            encounters_to_imaging_studies
            encounters_to_immunizations
            encounters_to_medications
            encounters_to_observations
            encounters_to_procedures
            encounters_to_supplies
            organizations_to_encounters
            organizations_to_providers
            patients_to_allergies
            patients_to_careplans
            patients_to_conditions
            patients_to_devices
            patients_to_encounters
            patients_to_imaging_studies
            patients_to_immunizations
            patients_to_medications
            patients_to_observations
            patients_to_procedures
            patients_to_supplies
            payers_to_encounters
            payers_to_medications
            providers_to_encounters

        Your job is to generate an AQL query based on the user's query. Make sure the query is correct and achieves the purpose and gets the info that is needed. Make sure the AQL query should work on the Synthea_P100 dataset and make sure that the collection names, field names, and any other thing you mention is compatible with the dataset that is in the ArangoDB dataset.
    """,

    # Prompt for the chunker tool
    "chunker_prompt":"""
        Your job is to take in the user query and break into the list of simple queries for easy retrieval. For instance, if the initial user queries ask about so many data from the db break those queries into simple queries with proper data so that at the end we can get collective data for the user queries for the AI model to use this to answer the user's initial query.
        Note: You will be dividing the chunks for the Synthea_P100 dataset stored in the ArangoDb. Your only job is to break the query into the list of simple queries.

        Keep in mind your job is not to create any queries but rather conver the user query into simple natural queries as if you are the human who is modifying the complex natural language queries into simple natural language queries for the further tools to act upon.
    """,

}

#### Tools for the AQL Agent

In [31]:
# Breaks the complex user query into smaller simpler queries(questions)
def chunker(user_query:str)->list[str]:
  """
  The tool is used to divide the user query into simple chunks(queries) for further tools

  Args:
  user_query: The initial user_query entered by the user

  Returns:
  The list of strings that contains queries for further tools are sent
  """
  print("================= USING THE chunker TOOL =================")
  json_repsonse = gemini_client.models.generate_content(
      model=GEMINI_FLASH_MODEL,
      config={
          "system_instruction": aql_tool_system_prompts["chunker_prompt"],
          "response_mime_type": "application/json",
          "response_schema":list[str]
      },
      contents=[user_query]
  )
  response = json.loads(json_repsonse.text)
  print(f"================= The new query is =================\n{response}")
  return response


# Generates aql queries
def aql_generator(user_query: str) -> dict[str, str]:
  """
  This is used to generate the AQL queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' and AQL query which will be a string
  """
  print("================= USING THE aql_generator TOOL =================")
  response = gemini_client.models.generate_content(
      model=GEMINI_PRO_MODEL,
      config={
          "system_instruction": aql_tool_system_prompts["aql_generator"],
          "response_mime_type": "application/json",
          "response_schema":AQLSchema
      },
      contents=[user_query]
  )
  return (json.loads(response.text))


def aql_executor(aql_query:str):
  """
  This is used to executed the queries that are in the AQL format using the arangoDB client

  Args:
    aql_query: This is the AQL query that can be executed in the ArangoDB

  Returns:
    This returns the list of the data from the ArangoDB. The empty list means that nothing related to that query is available
  """
  print("================= USING THE AQL_executor TOOL =================")
  print(f"AQL Query:\n{aql_query}")
  cursor = arango_client.aql.execute(aql_query)
  return list(cursor)



#### Function Definition for the AQL Agent



In [32]:
def aql_agent(user_query:str):
    print("====================== aql_agent ====================== \n\n")

    # Agent tools
    tools = [aql_generator, aql_executor,chunker]

    # Agent config
    config = {
        'tools': tools,
        'system_instruction': prompts_dict["aql_agent_prompt"],
    }

    try:
        agent_response = gemini_client.models.generate_content(
        model=GEMINI_FLASH_MODEL,
        config=config,
        contents=[user_query]
        )
        return agent_response.text

    except Exception as e:
        print("EXCEPTION HAS OCCURED IN aql_agent :: Retrying again")
        print(e)

### NetworkX Agent

#### NetworkX Specialised Agents

##### Degree Centrality Specialised Agent

In [114]:
# Tool system prompts
degree_centrality_prompts = {
    "generate_degree_centrality_aql_tool_prompt":"""
        You are an expert in ArangoDB and AQL. Your task is to generate an AQL query for calculating degree centrality on different node types within the Synthea_P100 dataset. The dataset includes the following vertex and edge collections:

        **Vertex Collections:**
            - allergies
            - careplans
            - conditions
            - devices
            - encounters
            - imaging_studies
            - immunizations
            - medications
            - observations
            - organizations
            - patients
            - payers
            - procedures
            - providers
            - supplies

        **Edge Collections:**
            - encounters_to_allergies
            - encounters_to_careplans
            - encounters_to_conditions
            - encounters_to_devices
            - encounters_to_imaging_studies
            - encounters_to_immunizations
            - encounters_to_medications
            - encounters_to_observations
            - encounters_to_procedures
            - encounters_to_supplies
            - organizations_to_encounters
            - organizations_to_providers
            - patients_to_allergies
            - patients_to_careplans
            - patients_to_conditions
            - patients_to_devices
            - patients_to_encounters
            - patients_to_imaging_studies
            - patients_to_immunizations
            - patients_to_medications
            - patients_to_observations
            - patients_to_procedures
            - patients_to_supplies
            - payers_to_encounters
            - payers_to_medications
            - providers_to_encounters

        Your only job is to generate correct AQL queries that could get the data in the proper format so that it could be used for the NetworkX degree_centrality algorithm
    """,


}

In [136]:
# Degree Centrality Agent tools
def generate_degree_centrality_aql(user_query:str) -> str:
    """
    The tool generates AQL queries for the query that needs to be executed using degree centrality.

    Args:
    user_query: The user query asked by the user

    Returns:
    String containing the AQL query that when executed gets the data in the form that is best suited to apply degree centrality graph algorithm.
    """
    print("==================== Executing generate_degree_centrality_aql Tool ====================")
    tool_response = gemini_client.models.generate_content(
        model=GEMINI_PRO_MODEL,
        config={
            "system_instruction":degree_centrality_prompts["generate_degree_centrality_aql_tool_prompt"]
        },
        contents=[user_query]
    )
    print("==================== Response : generate_degree_centrality_aql Tool ====================")
    print(tool_response.text)
    return tool_response.text

def degree_centrality_aql_executor(aql_query:str):
    """
    The tool is used to execute the AQL query generated by the previous tool(s) to get the data from ArangoDB for the networkX algorithms to work upon and finally answer the user's question.

    Args:
    aql_query: This will be a string containing the AQL query generated by the previous tool(s) to extract the data from ArangoDB

    Returns:
    The data extracted from ArangoDB to be used by the networkX graph algorithms.
    """
    print("==================== Executing degree_centrality_aql_executor Tool ====================")
    try:
        cursor = arango_client.aql.execute(aql_query)
        print("==================== Response : degree_centrality_aql_executor Tool :  First 5 retrieved data ====================")
        print(list(cursor)[:5])
        return list(cursor)
    except Exception as e:
        print(f"Error in degree_centrality_aql_executor :: {e}")
        return "Some error has occured"

In [137]:
def degree_centrality_agent(user_query:str):
    """Specialised agent to handle user_queries that would be best solved using degree_centrality graph algo"""
    tools = [generate_degree_centrality_aql,degree_centrality_aql_executor,]
    config={
        'tools': tools,
        'system_instruction': """You are the degree_centrality_agent who is specialised in using the given tools to get the AQL query, executes it and then use the proper tools assigned to you to answer the user question. You will not modify any response or anything from your end and will only use the tools given to perform the task of answering the user query. If there is a lack of tool you will reply the user with a message that why you can not help the user. At the end, once everything is fetched you will use the collected data to reply the user in the natural language.

        For now since you only have two tools, so once you have retrieved the AQL data after executing return that for now.

        """,
    }
    degree_centrality_agent_response = gemini_client.models.generate_content(
        model=GEMINI_FLASH_MODEL,
        config=config,
        contents=[user_query]
    )
    return degree_centrality_agent_response.text

#### NetworkX Agent Starter Function

In [138]:
class GraphAlgorithm(str, Enum):
    DEGREE_CENTRALITY = "degree_centrality"
    BETWEENNESS_CENTRALITY = "betweenness_centrality"
    LABEL_PROPAGATION = "label_propagation"
    LOUVAIN_MODULARITY = "louvain_modularity"
    SHORTEST_PATH = "shortest_path"
    NONE = "none"

class AlgoSelectionSchema(BaseModel):
    suggested_algo: GraphAlgorithm

# Checks the best graph algorithm to use and call the specific mini agent for that
def networkX_agent(user_query:str):
    config = {
        "response_mime_type": "application/json",
        "response_schema": AlgoSelectionSchema,
        "system_instruction":"Tell me what would be the best graph algorithm to choose for the given user_query so that the specific query to go through the networkX algorithms."
    }
    json_graph_algo_reponse = gemini_client.models.generate_content(
        model=GEMINI_FLASH_MODEL,
        config=config,
        contents=[user_query]
    )
    graph_algo_reponse = json.loads(json_graph_algo_reponse.text)
    match graph_algo_reponse["suggested_algo"]:
        case "degree_centrality":
            print("Using the degree_centrality specialised agent for the given user query")
            return degree_centrality_agent(user_query=user_query)
        case "betweenness_centrality":
            pass
        case "label_propagation":
            pass
        case "louvain_modularity":
            pass
        case "shortest_path":
            pass

### Takes the user input, checks the query, and suggest proper method for the answer to the user query

In [141]:
# Which medications are prescribed most frequently? using netwrokX the method
# Runs until a user query has been mentioned by the user
agent_memory = []
while True:
    user_query = input("Enter your query: ").strip()
    if not user_query:
        break
    else:
        starter_response = starter_function(user_query=user_query)
        if not starter_response["is_agent_recommended"]:
            print(starter_response["not_relevant_message"], end="\n\n")
            continue
        elif (starter_response["is_relevant"]):
            match starter_response["agent_choice"]:
                case "aql":
                    print(f"This query will go through AQL as agent choice is: {starter_response['agent_choice']}")
                    aql_agent_reponse = aql_agent(user_query=user_query)
                    print(aql_agent_reponse)
                    # Updating the agent memory
                    continue
                case "networkX":
                    print(f"This query will go through networkX as agent choice is: {starter_response['agent_choice']}")
                    networkX_agent_response = networkX_agent(user_query=user_query)
                    print(networkX_agent_response)
                    continue
                case "hybrid":
                    print(f"This query will go through hybrid method as agent choice is: {starter_response['agent_choice']}")
                    continue
                case _:
                    print(f"Invalid agent choice: {starter_response}")
                    continue
        else:
            print("Unexpected: ", starter_response)

Enter your query: # Which medications are prescribed most frequently? using netwrokX the method
================= Currently at : : Starter Function =====================

This query will go through networkX as agent choice is: networkX
Using the degree_centrality specialised agent for the given user query
==================== Executing generate_degree_centrality_aql Tool ====================
==================== Response : generate_degree_centrality_aql Tool ====================
This question asks about frequency of medication prescriptions, not degree centrality.  Here's the AQL to answer this:

```aql
FOR m IN medications
  LET prescription_count = COUNT(
    FOR e IN patients_to_medications
      FILTER e._to == m._id
      RETURN e
  )
  SORT prescription_count DESC
  RETURN { medication: m.DESCRIPTION, count: prescription_count }
```

==================== Executing degree_centrality_aql_executor Tool ====================
==================== Response : degree_centrality_aql_execut

KeyboardInterrupt: Interrupted by user

In [142]:
cursor = arango_client.aql.execute("""
FOR m IN medications
  LET prescription_count = COUNT(
    FOR e IN patients_to_medications
      FILTER e._to == m._id
      RETURN e
  )
  SORT prescription_count DESC
  RETURN { medication: m.DESCRIPTION, count: prescription_count }

""")

list(cursor)[:5]

[{'medication': 'Abuse-Deterrent 12 HR Oxycodone Hydrochloride 10 MG Extended Release Oral Tablet [Oxycontin]',
  'count': 1},
 {'medication': '1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]',
  'count': 1},
 {'medication': '1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]',
  'count': 1},
 {'medication': '1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]',
  'count': 1},
 {'medication': '1 ML Epoetin Alfa 4000 UNT/ML Injection [Epogen]',
  'count': 1}]